In [20]:
from pathlib import Path
import torch
import esm
import biotite.structure.io as bsio
import pandas as pd
from proteinttt.models.esmfold import ESMFoldTTT, DEFAULT_ESMFOLD_TTT_CFG

# Feb 2026

In [21]:
res_dir = Path("/scratch/project/open-35-8/pimenol1/ProteinTTT/ProteinTTT_fresh/data/bfvd/bfvd_sample_best/")
dir_esmfold = res_dir / "predicted_structures" / "ESMFold"
dir_proteinttt = res_dir / "predicted_structures" / "ESMFold_ProteinTTT"
df_bfvd = pd.read_csv(res_dir / "results.csv", sep=",")
df_bfvd

,id,nmsa,version,sequence,pLDDT_ESMFold,pLDDT_ProteinTTT,pLDDT_ColabFold,sequence_length,lddt_ProteinTTT,tm_score_ProteinTTT,lddt_ESMFold,tm_score_ESMFold
0,A0A059T6Q4,19,BASE+LOGAN+12CY,MKEKCSIEISYYNGEDKLTGKVVWVVLDKEEGIQFVLTEKPEDEYF...,50.321796,75.635233,42.065870,88,NaN,NaN,NaN,NaN
1,A0A060AHT0,91,BASE,MKQEIESIKIEFTFGEMSWDGCYVTFQQDWSDYSWFRSENSFVTSS...,46.781654,85.521093,44.442511,81,NaN,NaN,NaN,NaN
2,A0A075M4K8,24,BASE+LOGAN+12CY,MIPEVGKQYFGYLPVAMENYEEPYILVYSHKFESEYNGKTYWVFDA...,58.193550,74.748096,37.893749,166,NaN,NaN,NaN,NaN
3,A0A0A0PZG7,4,BASE+LOGAN+12CY,MKIQELLSQLKKFAEFKEFYGFVVNPSGSFSQSYTLDACMIADIKE...,61.730103,78.435108,45.662976,108,NaN,NaN,NaN,NaN
4,A0A0F6R7R7,5,BASE+LOGAN+12CY,MYRVYRRIKNPHKSELESRKVGYIFRGLPTISGNVAEIDEFEETVR...,44.556984,88.185425,50.080709,58,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
305,UPI001F1377E3,1386,BASE,MNDKIYFVTILVKQFSKLLYEIETQLLLARICSTESVNKKQLQITI...,83.788234,83.788234,41.072215,292,NaN,NaN,NaN,NaN
306,UPI0023292547,11,BASE+LOGAN+12CY,MVGYSREMWEVLMGLPSTSRSDAVNLSFAKFGHIIKAWVEGCNIEI...,37.487404,94.512179,38.947581,121,NaN,NaN,NaN,NaN
307,W6JLA7,97,BASE,MFNSKIITDNFNMSYKIMNNNLNINFNKIYNSTHKIDVLDKCFVIY...,34.318183,69.363117,34.711188,251,NaN,NaN,NaN,NaN
308,W8CZD8,41,BASE,MYSFIQEKKVMYNNFDNKKKLIYVQMAAATMQDLLNLTERAKTTAL...,49.529719,69.337458,34.425733,229,NaN,NaN,NaN,NaN


In [22]:
df_teo = pd.read_csv("/scratch/project/open-35-8/antonb/ttt/ProteinTTT/data/teo/Top_Candidate_TTT.txt", sep="\t")
df_teo["id"] = df_teo["Prot"].str.split("_").str[0]
# Merge df_bfvd into df_teo on column "id", bringing in specified columns
merge_cols = ["id", "nmsa", "pLDDT_ESMFold", "pLDDT_ProteinTTT", "pLDDT_ColabFold"]
df_teo = pd.merge(df_teo, df_bfvd[merge_cols], on="id", how="left")
df_teo["pLDDT_ProteinTTT - pLDDT_ESMFold"] = df_teo["pLDDT_ProteinTTT"] - df_teo["pLDDT_ESMFold"]

df_teo


,Prot,Foldseek homolog,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,CLEAN annotation,Unnamed: 7,Agreement,maxsep_top1_ec,...,NoveltyScore_0to10,FeasibilityScore_0to10,EvidenceConfidence_0to1,PredictedCategory,id,nmsa,pLDDT_ESMFold,pLDDT_ProteinTTT,pLDDT_ColabFold,pLDDT_ProteinTTT - pLDDT_ESMFold
0,A0A6J5RXK7_A,Nascent polypeptide complex,NaN,NaN,EC:5.4.99.4,0.0017,EC 5.4.99.5 (distance 0.0008): chorismate muta...,NaN,no,EC:5.4.99.4,...,10,10,0.974,aromatic_aa_biosynth,A0A6J5RXK7,237,84.091550,86.426951,52.110596,2.335400
1,A0A6M3YL53_A,tRNA thiolase,NaN,NaN,EC:1.12.1.2,0.0135,EC 1.12.1.2 (distance 0.0001): hydrogen dehydr...,NaN,no,EC:1.12.1.2,...,5,8,0.797,translation_related,A0A6M3YL53,3,87.581146,87.581146,44.067188,0.000000
2,A0A0A0PZG7_A,UPF0134 protein,NaN,NaN,EC:2.5.1.83,0.0001,EC 2.5.1.83 (distance 0.0001): hexaprenyl-diph...,NaN,no,EC:2.5.1.83,...,5,8,0.998,other_or_unclear,A0A0A0PZG7,4,61.730103,78.435108,45.662976,16.705006
3,A0A0U4IQ41_A,Unclear function,NaN,NaN,EC:2.1.1.86,0.0230,Methyl transferase,NaN,no,EC:2.1.1.86,...,4,7,0.700,other_or_unclear,A0A0U4IQ41,4,73.183200,79.778736,46.510100,6.595536
4,A0A1V0SB57_A,Glutamyl-tRNA reductase,NaN,NaN,EC:5.4.99.5,0.0128,EC 5.4.99.5 (distance 0.0008): chorismate muta...,NaN,no,EC:5.4.99.5,...,5,8,0.808,translation_related,A0A1V0SB57,63,70.243154,81.974876,45.513693,11.731722
5,A0A6J7WFZ2_A,"Enzymatic, unclear susbstract",NaN,NaN,EC:4.2.2.7,0.0000,EC 4.2.2.7 (distance 0.0000): heparin lyase. |...,NaN,no,EC:4.2.2.7,...,5,8,1.000,other_or_unclear,A0A6J7WFZ2,9,60.252242,87.905096,50.881623,27.652853
6,A0A8S5LWU4_A,Structural or trna ligase,NaN,NaN,EC:4.3.2.2,0.0016,EC 4.3.2.2 (distance 0.0016): adenylosuccinate...,NaN,no,EC:4.3.2.2,...,6,9,0.976,translation_related,A0A8S5LWU4,47,56.550895,88.123978,46.298609,31.573083
7,B3FIV3_A,Ribosomal protein,NaN,NaN,EC:1.3.1.91,0.0001,t rna synthase,NaN,no,EC:1.3.1.91,...,5,8,0.976,other_or_unclear,B3FIV3,36,45.162235,83.311458,37.567360,38.149223
8,X5I2M5_A,tyrosine phosphatase,NaN,NaN,EC:1.14.13.130,0.0001,EC 1.14.13.130 (distance 0.0003): pyrrole-2-ca...,NaN,no,EC:1.14.13.130,...,9,10,0.908,aromatic_aa_biosynth,X5I2M5,3,55.783633,84.800817,51.324650,29.017183
9,A0A2H4UUY1_A,mRNA-capping enzyme subunit beta,NaN,NaN,EC:3.1.21.4,0.0875,EC 3.1.21.4 (distance 0.0875): type II site-sp...,NaN,no,EC:3.1.21.4,...,2,8,0.592,mrna_capping,A0A2H4UUY1,112,33.916898,78.850062,33.684105,44.933164


In [23]:
# Export df_teo to Excel with embedded structure images (ESMFold vs ProteinTTT)
#
# This uses a separate conda env (ttt-export) that contains PyMOL + xlsxwriter.
from pathlib import Path
import subprocess
import textwrap

# Where structures live
# - ProteinTTT structures are in: res_dir/predicted_structures/ESMFold_ProteinTTT
# - Some datasets (e.g. bfvd_sample_best) have *truncated* ESMFold PDBs (1 byte, content "P").
#   In that case, we fall back to bfvd_sample where full ESMFold predictions exist.
dir_esmfold = res_dir / "predicted_structures" / "ESMFold"
dir_proteinttt = res_dir / "predicted_structures" / "ESMFold_ProteinTTT"

try:
    esm_sizes = [p.stat().st_size for p in dir_esmfold.glob("*.pdb")]
    if esm_sizes and max(esm_sizes) < 1000:
        fallback = res_dir.parent / "bfvd_sample" / "predicted_structures" / "ESMFold"
        if fallback.exists():
            dir_esmfold = fallback
except Exception:
    pass

out_dir = Path("/scratch/project/open-35-8/antonb/ttt/ProteinTTT/outputs/teo_excel_export")
img_dir = out_dir / "images"
out_dir.mkdir(parents=True, exist_ok=True)
img_dir.mkdir(parents=True, exist_ok=True)

csv_path = out_dir / "df_teo.csv"
xlsx_path = out_dir / "df_teo_with_structures.xlsx"
script_path = out_dir / "export_teo_excel_with_images.py"

df_teo.to_csv(csv_path, index=False)

script_path.write_text(
    textwrap.dedent(
        r'''
        import argparse
        from pathlib import Path

        import pandas as pd
        from PIL import Image, ImageDraw


        def find_structure(struct_dir: Path, protein_id: str) -> Path | None:
            if not struct_dir.exists():
                return None

            # Expected filenames look like: {id}_A.pdb
            hits = sorted(struct_dir.glob(f"{protein_id}_*.pdb"))
            if hits:
                return hits[0]

            # Fallbacks
            hits = sorted(struct_dir.glob(f"{protein_id}*.pdb"))
            if hits:
                return hits[0]

            hits = sorted(struct_dir.rglob(f"{protein_id}_*.pdb"))
            if hits:
                return hits[0]

            hits = sorted(struct_dir.rglob(f"{protein_id}*.pdb"))
            if hits:
                return hits[0]

            return None


        def write_placeholder_png(out_png: Path, label: str, size: int = 350) -> None:
            out_png.parent.mkdir(parents=True, exist_ok=True)
            im = Image.new("RGBA", (size, size), (255, 255, 255, 255))
            d = ImageDraw.Draw(im)
            d.rectangle([0, 0, size - 1, size - 1], outline=(180, 180, 180, 255), width=2)
            d.text((12, 12), "MISSING", fill=(200, 0, 0, 255))
            d.text((12, 40), label, fill=(0, 0, 0, 255))
            im.save(out_png)


        def pymol_render_png(cmd, pdb_path: Path, out_png: Path, size: int = 350) -> None:
            out_png.parent.mkdir(parents=True, exist_ok=True)

            cmd.delete("all")
            cmd.load(str(pdb_path), "m")
            cmd.hide("everything", "all")
            cmd.show("cartoon", "all")
            cmd.bg_color("white")
            cmd.set("ray_opaque_background", 0)

            # pLDDT is stored in B-factor; apply your discrete threshold palette
            cmd.set_color("n0", [0.051, 0.341, 0.827])
            cmd.set_color("n1", [0.416, 0.796, 0.945])
            cmd.set_color("n2", [0.996, 0.851, 0.212])
            cmd.set_color("n3", [0.992, 0.490, 0.302])
            cmd.color("n0", "b < 100")
            cmd.color("n1", "b < 90")
            cmd.color("n2", "b < 70")
            cmd.color("n3", "b < 50")

            cmd.orient("all")
            cmd.zoom("all")

            cmd.png(str(out_png), width=size, height=size, dpi=150, ray=1)


        def add_images_to_excel(df: pd.DataFrame, out_xlsx: Path, esm_imgs: list[Path], ttt_imgs: list[Path]) -> None:
            df_out = df.copy()
            col_esm = "ESMFold_structure"
            col_ttt = "ProteinTTT_structure"
            df_out[col_esm] = ""
            df_out[col_ttt] = ""

            with pd.ExcelWriter(out_xlsx, engine="xlsxwriter") as writer:
                df_out.to_excel(writer, index=False, sheet_name="teo")
                ws = writer.sheets["teo"]

                # Layout: make room for images
                img_col_width = 22
                img_row_height = 140

                esm_col_idx = df_out.columns.get_loc(col_esm)
                ttt_col_idx = df_out.columns.get_loc(col_ttt)
                ws.set_column(esm_col_idx, esm_col_idx, img_col_width)
                ws.set_column(ttt_col_idx, ttt_col_idx, img_col_width)

                # Data starts on row 1 (row 0 is the header)
                for i, (esm_png, ttt_png) in enumerate(zip(esm_imgs, ttt_imgs)):
                    excel_row = i + 1
                    ws.set_row(excel_row, img_row_height)

                    if esm_png.exists():
                        ws.insert_image(excel_row, esm_col_idx, str(esm_png), {"x_scale": 0.9, "y_scale": 0.9, "object_position": 1})
                    if ttt_png.exists():
                        ws.insert_image(excel_row, ttt_col_idx, str(ttt_png), {"x_scale": 0.9, "y_scale": 0.9, "object_position": 1})


        def main() -> None:
            ap = argparse.ArgumentParser()
            ap.add_argument("--csv", required=True)
            ap.add_argument("--esmfold-dir", required=True)
            ap.add_argument("--proteinttt-dir", required=True)
            ap.add_argument("--out-xlsx", required=True)
            ap.add_argument("--img-dir", required=True)
            args = ap.parse_args()

            csv_path = Path(args.csv)
            esm_dir = Path(args.esmfold_dir)
            ttt_dir = Path(args.proteinttt_dir)
            out_xlsx = Path(args.out_xlsx)
            img_dir = Path(args.img_dir)
            img_dir.mkdir(parents=True, exist_ok=True)

            df = pd.read_csv(csv_path)
            if "id" not in df.columns:
                raise ValueError("Expected an 'id' column in df_teo")

            # Launch PyMOL once
            import pymol

            pymol.finish_launching(["pymol", "-cq"])
            from pymol import cmd  # noqa: E402

            esm_imgs: list[Path] = []
            ttt_imgs: list[Path] = []

            for protein_id in df["id"].astype(str).tolist():
                esm_png = img_dir / f"{protein_id}__ESMFold.png"
                ttt_png = img_dir / f"{protein_id}__ProteinTTT.png"

                esm_pdb = find_structure(esm_dir, protein_id)
                ttt_pdb = find_structure(ttt_dir, protein_id)

                if esm_pdb is None:
                    write_placeholder_png(esm_png, f"ESMFold: {protein_id}")
                else:
                    pymol_render_png(cmd, esm_pdb, esm_png)

                if ttt_pdb is None:
                    write_placeholder_png(ttt_png, f"ProteinTTT: {protein_id}")
                else:
                    pymol_render_png(cmd, ttt_pdb, ttt_png)

                esm_imgs.append(esm_png)
                ttt_imgs.append(ttt_png)

            add_images_to_excel(df, out_xlsx, esm_imgs, ttt_imgs)

            cmd.quit()


        if __name__ == "__main__":
            main()
        '''
    ).lstrip()
)

# Ensure the export env exists (PyMOL + xlsxwriter). Creating it is a one-time step.
import json

def _ensure_conda_env(env_name: str = "ttt-export") -> None:
    envs_json = subprocess.run(
        ["conda", "env", "list", "--json"],
        check=True,
        capture_output=True,
        text=True,
    )
    env_paths = json.loads(envs_json.stdout).get("envs", [])
    if any(Path(p).name == env_name for p in env_paths):
        return

    print(f"Conda env '{env_name}' not found; creating it (this can take a few minutes)...")
    subprocess.run(
        [
            "conda",
            "create",
            "-y",
            "-n",
            env_name,
            "-c",
            "conda-forge",
            "python=3.11",
            "pymol-open-source",
            "pandas",
            "xlsxwriter",
            "pillow",
        ],
        check=True,
    )


_ensure_conda_env("ttt-export")

cmd = [
    "conda",
    "run",
    "-n",
    "ttt-export",
    "python",
    str(script_path),
    "--csv",
    str(csv_path),
    "--esmfold-dir",
    str(dir_esmfold),
    "--proteinttt-dir",
    str(dir_proteinttt),
    "--out-xlsx",
    str(xlsx_path),
    "--img-dir",
    str(img_dir),
]

print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)
print("Wrote:", xlsx_path)


Running: conda run -n ttt-export python /scratch/project/open-35-8/antonb/ttt/ProteinTTT/outputs/teo_excel_export/export_teo_excel_with_images.py --csv /scratch/project/open-35-8/antonb/ttt/ProteinTTT/outputs/teo_excel_export/df_teo.csv --esmfold-dir /scratch/project/open-35-8/pimenol1/ProteinTTT/ProteinTTT_fresh/data/bfvd/bfvd_sample/predicted_structures/ESMFold --proteinttt-dir /scratch/project/open-35-8/pimenol1/ProteinTTT/ProteinTTT_fresh/data/bfvd/bfvd_sample_best/predicted_structures/ESMFold_ProteinTTT --out-xlsx /scratch/project/open-35-8/antonb/ttt/ProteinTTT/outputs/teo_excel_export/df_teo_with_structures.xlsx --img-dir /scratch/project/open-35-8/antonb/ttt/ProteinTTT/outputs/teo_excel_export/images
Wrote: /scratch/project/open-35-8/antonb/ttt/ProteinTTT/outputs/teo_excel_export/df_teo_with_structures.xlsx


# Dec 2025

In [ ]:
# Set your sequence
sequence = "GIHLGELGLLPSTVLAIGYFENLVNIICESLNMLPKLEVSGKEYKKFKFTIVIPKDLDANIKKRAKIYFKQKSLIEIEIPTSSRNYPIHIQFDENSTDDILHLYDMPTTIGGIDKAIEMFMRKGHIGKTDQQKLLEERELRNFKTTLENLIATDAFAKEMVEVIIEE"

# Load model
model = esm.pretrained.esmfold_v1()
model = model.eval().cuda()

def predict_structure(model, sequence):
    with torch.no_grad():
        output = model.infer_pdb(sequence)

    with open("result.pdb", "w") as f:
        f.write(output)

    struct = bsio.load_structure("result.pdb", extra_fields=["b_factor"])
    print('pLDDT:', struct.b_factor.mean())

predict_structure(model, sequence)
# pLDDT: 38.43025

# ============ ProteinTTT =============
ttt_cfg = DEFAULT_ESMFOLD_TTT_CFG
model = ESMFoldTTT.ttt_from_pretrained(model, ttt_cfg=ttt_cfg, esmfold_config=model.cfg)
model.ttt(sequence)
# =====================================

predict_structure(model, sequence)
# pLDDT: 78.69619

# Reset model to original state (after this model.ttt can be called again on another protein)
# ============== ProteinTTT ===========
model.ttt_reset()
# =====================================

/scratch/project/open-35-8/antonb/miniconda3/envs/esmfold/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/scratch/project/open-35-8/antonb/miniconda3/envs/esmfold/lib/python3.10/site-packages/lightning_lite/__init__.py:29: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)
/scratch/project/open-35-8/antonb/miniconda3/envs/esmfold/lib/python3.10/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used

pLDDT: 38.430245170876674
2026-01-13 19:14:19,168 | INFO | step: 0, accumulated_step: 0, loss: None, perplexity: None, ttt_step_time: 0.00000, score_seq_time: 0.00000, eval_step_time: 1.76557, plddt: 38.43025
2026-01-13 19:14:21,512 | INFO | step: 1, accumulated_step: 4, loss: 2.54688, perplexity: None, ttt_step_time: 0.58051, score_seq_time: 0.00000, eval_step_time: 1.76197, plddt: 33.03000
2026-01-13 19:14:23,787 | INFO | step: 2, accumulated_step: 8, loss: 2.52148, perplexity: None, ttt_step_time: 0.51777, score_seq_time: 0.00000, eval_step_time: 1.75736, plddt: 34.36354
2026-01-13 19:14:26,063 | INFO | step: 3, accumulated_step: 12, loss: 2.48242, perplexity: None, ttt_step_time: 0.51855, score_seq_time: 0.00000, eval_step_time: 1.75633, plddt: 36.54465
2026-01-13 19:14:28,653 | INFO | step: 4, accumulated_step: 16, loss: 2.28516, perplexity: None, ttt_step_time: 0.51934, score_seq_time: 0.00000, eval_step_time: 1.75659, plddt: 77.43795
2026-01-13 19:14:30,937 | INFO | step: 5, acc

In [5]:
uniprot_ids = [
    "G9E3N8",
    "G4YD72",
    "G3GP93"
]
sequences = [
    "MRVVVVMVVVVMVVVDLEVVVTAVVVVMVVARVVVDLEVVVTVVVRVEVAKAVVVMAVVVMAEGMVAEEKGEVMAGDLVVVVRAAADLAVVGLVAVVMVVEETVVVAMVVVETAVEGTVEVMVVGLVVVVTVVAGTVVAGTVVAGTVVVVMVAVMAAAVMVVVEMVVVGMAVVMVVVMVAVVTEEGLVVAMVVAVTEEGLVVEMVVVVTAAVMAVVVMAVVVMAVVVMAAGEKVIYQSE",
    "MRVVVVMVVVVMVVVDLEVVVTAVVVMVVVARVVVDLEVVVTVVVRVEVAKAVVVMAVVVMAEGMVAEEKGEVMAGDLVVVVRAAADLAVVGLVAVVMVVEETVVVAMVVVETAVEGTVEVMVVGLVVVVTVVAGTVVAGTVVAGTVVVVMVAVMAAAVMVVVEMVVVGMAVVMVVVMVAVVTEEGLVVAMVVAVTEEGLVVEMVVVVTAAVMAVVVMAVVVMAVVVMAAGEKVIYQSE",
    "MVAVAMAVVAMVVVDLVVVMAVAVMAAVDLAVAVMVVVMAAVAMAAVDLAAAVTVVAATVAVAMAMAMAMADLAVAEMVAVAMAATMVVAVTVAVTMAAVMAAAVMAVAVMAVGRWRWRWWRWWLGDGGGVMAVGVTAEAVMAVVEMVAAVMVVEAMVEVTVAVTVAADLVVVDLVVDLVEAVMVVAEGLGDNM"
]

In [6]:
ttt_cfg = DEFAULT_ESMFOLD_TTT_CFG
for uniprot_id, sequence in zip(uniprot_ids, sequences):
    for seed in range(3):
        print('uniprot_id', uniprot_id)
        print('sequence', sequence)
        print('seed', seed)
        ttt_cfg.seed = seed
        model.ttt_reset()
        model = ESMFoldTTT.ttt_from_pretrained(model, ttt_cfg=ttt_cfg, esmfold_config=model.cfg)
        model.ttt(sequence)
        predict_structure(model, sequence)

uniprot_id G9E3N8
sequence MRVVVVMVVVVMVVVDLEVVVTAVVVVMVVARVVVDLEVVVTVVVRVEVAKAVVVMAVVVMAEGMVAEEKGEVMAGDLVVVVRAAADLAVVGLVAVVMVVEETVVVAMVVVETAVEGTVEVMVVGLVVVVTVVAGTVVAGTVVAGTVVVVMVAVMAAAVMVVVEMVVVGMAVVMVVVMVAVVTEEGLVVAMVVAVTEEGLVVEMVVVVTAAVMAVVVMAVVVMAVVVMAAGEKVIYQSE
seed 0
2026-01-13 19:27:39,527 | INFO | step: 0, accumulated_step: 0, loss: None, perplexity: None, ttt_step_time: 0.00000, score_seq_time: 0.00000, eval_step_time: 3.25016, plddt: 32.21878
2026-01-13 19:27:43,381 | INFO | step: 1, accumulated_step: 4, loss: 0.93555, perplexity: None, ttt_step_time: 0.60923, score_seq_time: 0.00000, eval_step_time: 3.24422, plddt: 32.17507
2026-01-13 19:27:47,235 | INFO | step: 2, accumulated_step: 8, loss: 0.95508, perplexity: None, ttt_step_time: 0.61055, score_seq_time: 0.00000, eval_step_time: 3.24269, plddt: 31.73399
2026-01-13 19:27:51,089 | INFO | step: 3, accumulated_step: 12, loss: 1.10254, perplexity: None, ttt_step_time: 0.61004, score_seq_time: 0.00000, eval_step_time: 3.24307, 